# SQLite Extraction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakelogic/LakeLogic/blob/main/examples/03_data_sources/databases/sqlite/sqlite_demo.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/03_data_sources/databases/sqlite/sqlite_demo.ipynb)

## Business Scenario

Local SQLite databases are common in prototypes and edge apps. You need a quick extraction flow.

## Value Proposition

- Minimal setup for local extraction
- Contract-based schema enforcement
- Simple validation for small datasets

---

## Goals

1. Define a SQLite contract
2. Run extraction
3. Validate output


## Step 2: Review the Contract

Open `sqlite_contract.yaml` to see the schema, source config, and quality rules.


## Step 3: Run the Contract

Run LakeLogic using the YAML contract. Update credentials as needed.


In [ ]:
from pathlib import Path
import sqlite3
from lakelogic import DataProcessor

BASE = Path.cwd()
contract_path = BASE / "sqlite_contract.yaml"
if not contract_path.exists():
    candidate = BASE / "examples" / "03_data_sources" / "databases" / "sqlite" / "sqlite_contract.yaml"
    if candidate.exists():
        BASE = candidate.parent
        contract_path = candidate

db_path = BASE / "example.db"

if not db_path.exists():
    conn = sqlite3.connect(db_path)
    conn.execute("CREATE TABLE users (id INTEGER, name TEXT, email TEXT)")
    conn.executemany(
        "INSERT INTO users (id, name, email) VALUES (?, ?, ?)",
        [
            (1, "Ava", "ava@example.com"),
            (2, "Ben", ""),
            (3, "Cara", "cara@example.com"),
        ],
    )
    conn.commit()
    conn.close()

conn = sqlite3.connect(db_path)
rows = [
    {"id": row[0], "name": row[1], "email": row[2]}
    for row in conn.execute("SELECT id, name, email FROM users")
]
conn.close()

processor = DataProcessor(contract=contract_path)
result = processor.run(rows, source_path=str(db_path))

print(result)
print(f"Good: {len(result.good)} | Bad: {len(result.bad)}")
